# 🤖 Model Training & Evaluation Notebook

End-to-end walkthrough of all ML and DL models.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_collection.data_loader         import load_data
from src.feature_engineering.feature_engineer import FeatureEngineer
from src.preprocessing.preprocess             import DataPreprocessor
from src.models.train_model                   import ModelTrainer

sns.set_theme(style='darkgrid')
%matplotlib inline
print('Imports OK')

## 1. Load & Enrich Data

In [ ]:
df_raw      = load_data()
fe          = FeatureEngineer()
df_enriched = fe.transform(df_raw)
print(f'Raw shape: {df_raw.shape}  |  Enriched shape: {df_enriched.shape}')
df_enriched.head(3)

## 2. Preprocessing

In [ ]:
prep = DataPreprocessor()
X_train, X_test, y_train, y_test, feature_cols = prep.fit_transform(df_enriched)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print('Features:', feature_cols)

## 3. Train ML Models

In [ ]:
trainer = ModelTrainer()
results = trainer.train(X_train, X_test, y_train, y_test)
trainer.print_comparison_table()

## 4. Feature Importance

In [ ]:
fi = trainer.get_feature_importance(feature_cols)
if len(fi):
    fi.head(10).plot(kind='barh', x='feature', y='importance', color='steelblue', figsize=(10,5))
    plt.title(f'Feature Importances — {trainer.best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 5. Model Comparison Chart

In [ ]:
df_res = pd.DataFrame(results).T
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_res['r2'].sort_values().plot(kind='barh', ax=axes[0], color='#6c47ff')
axes[0].set_title('R² Score (higher = better)')
axes[0].set_xlim(0, 1)

df_res['mape'].sort_values().plot(kind='barh', ax=axes[1], color='#00d4a8')
axes[1].set_title('MAPE % (lower = better)')

plt.tight_layout()
plt.show()

## 6. Actual vs Predicted

In [ ]:
best_preds = trainer.best_model.predict(X_test)

plt.figure(figsize=(8, 6))
plt.scatter(y_test/1e6, best_preds/1e6, alpha=0.3, s=10, color='#6c47ff')
lim = [0, max(y_test.max(), best_preds.max())/1e6 * 1.05]
plt.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
plt.xlabel('Actual Price (₹M)')
plt.ylabel('Predicted Price (₹M)')
plt.title(f'Actual vs Predicted — {trainer.best_model_name}')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Investment Analysis Sample

In [ ]:
from src.investment_analysis.investment_analysis import InvestmentAnalyzer

analyzer = InvestmentAnalyzer()
report   = analyzer.analyze(price=8_500_000, city='Pune')
analyzer.print_report(report)

## 8. Anomaly Detection Sample

In [ ]:
from src.anomaly_detection.anomaly_detection import AnomalyDetector
import json

detector = AnomalyDetector(contamination=0.05)
detector.fit(df_enriched)
flagged = detector.detect(df_enriched)
stats   = detector.summary(flagged)
print(json.dumps(stats, indent=2))

# Show top anomalies
flagged[flagged['anomaly_flag']==1][['price','area_sqft','anomaly_score','anomaly_reason']].head(10)